In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D4 — Eurostat LFS Metadata Workbook
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import html
import json
import re
import unicodedata

import numpy as np
import pandas as pd

In [ ]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DOCUMENT_ID = "D4"
BRANCH_ID = "A"
EXPECTED_RECORD_COUNT = 83

REFERENCE_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted",
    "Source Location"
]

EXTRACTION_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

MATCHING_FIELDS = [
    "Section",
    "Concept Name"
]

PRIMARY_CORRECTNESS_FIELDS = [
    field
    for field in EXTRACTION_FIELDS
    if field not in MATCHING_FIELDS
]

ALLOWED_PUBLICATION_FLAGS = {"YES", "NO"}

OUTPUT_DIR = Path("outputs_D4_validation_branch_A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID)
print("Expected reference records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ------------------------------------------------------------
# 2. Validation input upload
# ------------------------------------------------------------
# Uploads:
#   1) D4_reference_values.csv
#   2) D4_branch_A_raw_response.txt
#   3) D4_branch_A_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
txt_files = [f for f in uploaded_files if f.lower().endswith(".txt")]
json_files = [f for f in uploaded_files if f.lower().endswith(".json")]

if len(csv_files) != 1:
    raise ValueError("Upload exactly one Stage 1 reference-values CSV.")

if len(txt_files) != 1:
    raise ValueError("Upload exactly one preserved Branch A raw-response TXT.")

if len(json_files) != 1:
    raise ValueError("Upload exactly one Branch A technical-diagnostics JSON.")

REFERENCE_FILE = csv_files[0]
RAW_RESPONSE_FILE = txt_files[0]
TECHNICAL_DIAGNOSTICS_FILE = json_files[0]

print("Reference:", REFERENCE_FILE)
print("Raw Branch A response:", RAW_RESPONSE_FILE)
print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_FILE
)

In [ ]:
# ------------------------------------------------------------
# 3. Input loading and provenance
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)

def restore_null(value):
    if value is None:
        return None
    if isinstance(value, str) and value == "":
        return None
    return value

for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_null)

raw_response_text = Path(RAW_RESPONSE_FILE).read_text(encoding="utf-8-sig")

with open(TECHNICAL_DIAGNOSTICS_FILE, "r", encoding="utf-8-sig") as f:
    technical_diagnostics = json.load(f)

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "raw_response_file": RAW_RESPONSE_FILE,
    "raw_response_sha256": sha256_file(RAW_RESPONSE_FILE),
    "technical_diagnostics_file": TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256": sha256_file(TECHNICAL_DIAGNOSTICS_FILE)
}

print("Reference shape:", reference_df.shape)
print("Raw response characters:", len(raw_response_text))
print(json.dumps(input_provenance, indent=2))


In [ ]:
# ------------------------------------------------------------
# 4. Fixed-reference verification
# ------------------------------------------------------------

reference_schema_valid = (
    reference_df.columns.tolist() == REFERENCE_FIELDS
)

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

if not reference_schema_valid:
    raise ValueError("The D4 Stage 1 reference schema is invalid.")

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} reference records, "
        f"found {len(reference_df)}."
    )

print("Reference schema valid:", reference_schema_valid)
print("Reference record count valid:", reference_record_count_valid)


In [ ]:
# ------------------------------------------------------------
# 5. Structural validity
# ------------------------------------------------------------

if technical_diagnostics.get("document_id") != DOCUMENT_ID:
    raise ValueError("Technical-diagnostics document_id does not match D4.")

if technical_diagnostics.get("branch") != BRANCH_ID:
    raise ValueError("Technical-diagnostics branch does not match Branch A.")

try:
    extraction_object = json.loads(raw_response_text)
    raw_json_valid = True
    raw_json_error = None
except json.JSONDecodeError as exc:
    extraction_object = None
    raw_json_valid = False
    raw_json_error = str(exc)

if raw_json_valid != bool(technical_diagnostics.get("valid_json", False)):
    raise ValueError(
        "The uploaded raw response does not reproduce the JSON-validity "
        "result recorded in the Branch A technical diagnostics."
    )

schema_validity = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

content_evaluable = bool(
    schema_validity
    and raw_json_valid
    and isinstance(extraction_object, dict)
    and extraction_object.get("document_id") == DOCUMENT_ID
    and extraction_object.get("branch") == BRANCH_ID
    and isinstance(
        extraction_object.get("records"),
        list
    )
)

schema_diagnostics = {
    "valid_json": raw_json_valid,
    "json_parsing_error": raw_json_error,
    "top_level_object_valid": bool(
        technical_diagnostics.get("top_level_object_valid", False)
    ),
    "document_id_correct": bool(
        technical_diagnostics.get("document_id_correct", False)
    ),
    "branch_correct": bool(
        technical_diagnostics.get("branch_correct", False)
    ),
    "records_is_list": bool(
        technical_diagnostics.get("records_is_list", False)
    ),
    "records_with_structure_issues": technical_diagnostics.get(
        "records_with_structure_issues"
    ),
    "records_with_type_issues": technical_diagnostics.get(
        "records_with_type_issues"
    ),
    "publication_flags_valid": technical_diagnostics.get(
        "publication_flags_valid"
    ),
    "schema_validity": schema_validity,
    "content_evaluable": content_evaluable
}

print(json.dumps(schema_diagnostics, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 6. Structural-failure result
# ------------------------------------------------------------

if not content_evaluable:
    structural_summary = {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH_ID,
        "reference_records": int(len(reference_df)),
        "extracted_records": None,
        "aligned_records": None,
        "fully_correct_records": None,
        "discrepant_records": None,
        "missing_records": None,
        "unsupported_extracted_records": None,
        "completeness": None,
        "missing_rate": None,
        "record_precision_exact": None,
        "record_recall_exact": None,
        "record_f1_exact": None,
        "unsupported_rate": None,
        "field_accuracy": None,
        "schema_validity": schema_validity,
        "content_evaluable": False,
        "validation_status": "Structural failure — content validation not evaluable",
        "schema_diagnostics": schema_diagnostics,
        "normalisation_note": (
            "No repair or content-level normalisation was applied because "
            "the preserved Branch A response is not structurally evaluable."
        ),
        "input_provenance": input_provenance
    }

    with open(
        OUTPUT_DIR / "D4_branch_A_validation_summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(structural_summary, f, indent=2, ensure_ascii=False)

    parsing_error_df = pd.DataFrame([{
        "document_id": DOCUMENT_ID,
        "branch": BRANCH_ID,
        "json_valid": raw_json_valid,
        "json_parsing_error": raw_json_error,
        "content_evaluable": False
    }])

    parsing_error_df.to_csv(
        OUTPUT_DIR / "D4_branch_A_structural_failure.csv",
        index=False
    )

    print(json.dumps(structural_summary, indent=2, ensure_ascii=False))
    print(
        "\nContent comparison is intentionally skipped. "
        "Do not replace this response with a manually corrected JSON file."
    )

In [ ]:
# ------------------------------------------------------------
# 7. Extraction preparation
# ------------------------------------------------------------

if content_evaluable:
    extracted_records = extraction_object["records"]
    extracted_df = pd.DataFrame(extracted_records)

    missing_extraction_fields = [
        field for field in EXTRACTION_FIELDS
        if field not in extracted_df.columns
    ]

    for field in missing_extraction_fields:
        extracted_df[field] = None

    extracted_df = extracted_df[EXTRACTION_FIELDS].copy()

    print("Extracted records:", len(extracted_df))
    print("Missing extraction columns:", missing_extraction_fields)
else:
    extracted_df = pd.DataFrame(columns=EXTRACTION_FIELDS)


In [ ]:
# ------------------------------------------------------------
# 8. Comparison normalisation
# ------------------------------------------------------------

UNICODE_SPACES = {
    "\u00a0": " ",
    "\u2007": " ",
    "\u202f": " "
}

APOSTROPHE_REPLACEMENTS = {
    "\u2018": "'",
    "\u2019": "'",
    "\u02bc": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-"
}

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")

def normalise_text(value):
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))

    for source, target in UNICODE_SPACES.items():
        text = text.replace(source, target)

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(source, target)

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(source, target)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip().casefold()

def normalise_html_content(value):
    if value is None:
        return None

    text = html.unescape(str(value))
    text = re.sub(r"</p\s*>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = HTML_TAG_PATTERN.sub("", text)

    return normalise_text(text)

def exact_equal(left, right):
    return left == right


In [ ]:
# ------------------------------------------------------------
# 9. Record identity construction
# ------------------------------------------------------------

if content_evaluable:
    ref_cmp = reference_df.copy(deep=True)
    ext_cmp = extracted_df.copy(deep=True)

    for df in (ref_cmp, ext_cmp):
        df["_matching_key"] = df.apply(
            lambda row: (
                normalise_text(row["Section"]),
                normalise_text(row["Concept Name"])
            ),
            axis=1
        )

    reference_duplicate_mask = ref_cmp["_matching_key"].duplicated(keep=False)
    if reference_duplicate_mask.any():
        raise ValueError(
            "The fixed Stage 1 reference contains duplicate matching keys."
        )

    extraction_duplicate_mask = ext_cmp["_matching_key"].duplicated(keep="first")
    duplicate_extracted_records = ext_cmp[
        extraction_duplicate_mask
    ].copy()

    ext_unique = ext_cmp[
        ~extraction_duplicate_mask
    ].copy()

    print("Reference duplicate keys:", int(reference_duplicate_mask.sum()))
    print(
        "Additional extracted duplicate records:",
        len(duplicate_extracted_records)
    )
else:
    ref_cmp = pd.DataFrame()
    ext_cmp = pd.DataFrame()
    ext_unique = pd.DataFrame()
    duplicate_extracted_records = pd.DataFrame()


In [ ]:
# ------------------------------------------------------------
# 10. One-to-one record alignment
# ------------------------------------------------------------

if content_evaluable:
    validation_df = ref_cmp.merge(
        ext_unique,
        on="_matching_key",
        how="outer",
        suffixes=("_ref", "_ext"),
        indicator=True,
        validate="one_to_one"
    )

    aligned_mask = validation_df["_merge"] == "both"
    print(validation_df["_merge"].value_counts(dropna=False))
else:
    validation_df = pd.DataFrame()
    aligned_mask = pd.Series(dtype=bool)

In [ ]:
# ------------------------------------------------------------
# 11. Field-level comparison
# ------------------------------------------------------------

if content_evaluable:
    validation_df["Section_match"] = False
    validation_df["Concept Name_match"] = False
    validation_df["Concept Value_match"] = False
    validation_df["Publication Restricted_match"] = False

    validation_df["Concept Value_normalised_match"] = False
    validation_df["Concept Value_content_match"] = False

    validation_df.loc[aligned_mask, "Section_match"] = (
        validation_df.loc[aligned_mask, "Section_ref"]
        == validation_df.loc[aligned_mask, "Section_ext"]
    )

    validation_df.loc[aligned_mask, "Concept Name_match"] = (
        validation_df.loc[aligned_mask, "Concept Name_ref"]
        == validation_df.loc[aligned_mask, "Concept Name_ext"]
    )

    validation_df.loc[aligned_mask, "Concept Value_match"] = (
        validation_df.loc[aligned_mask, "Concept Value_ref"]
        == validation_df.loc[aligned_mask, "Concept Value_ext"]
    )

    validation_df.loc[aligned_mask, "Publication Restricted_match"] = (
        validation_df.loc[aligned_mask, "Publication Restricted_ref"]
        == validation_df.loc[aligned_mask, "Publication Restricted_ext"]
    )

    validation_df.loc[aligned_mask, "Concept Value_normalised_match"] = (
        validation_df.loc[aligned_mask].apply(
            lambda row:
                normalise_text(row["Concept Value_ref"])
                == normalise_text(row["Concept Value_ext"]),
            axis=1
        )
    )

    validation_df.loc[aligned_mask, "Concept Value_content_match"] = (
        validation_df.loc[aligned_mask].apply(
            lambda row:
                normalise_html_content(row["Concept Value_ref"])
                == normalise_html_content(row["Concept Value_ext"]),
            axis=1
        )
    )

    PRIMARY_MATCH_COLUMNS = [
        "Concept Value_match",
        "Publication Restricted_match"
    ]

    validation_df["all_primary_fields_match"] = (
        aligned_mask
        & validation_df[
            PRIMARY_MATCH_COLUMNS
        ].all(axis=1)
    )

In [ ]:
# ------------------------------------------------------------
# 12. Record-outcome classification and metrics
# ------------------------------------------------------------

if content_evaluable:
    def classify_record(row):
        if row["_merge"] == "left_only":
            return "missing"
        if row["_merge"] == "right_only":
            return "unsupported_unmatched"
        if bool(row["all_primary_fields_match"]):
            return "fully_correct"
        return "discrepant"

    validation_df["record_status"] = validation_df.apply(
        classify_record,
        axis=1
    )

    missing_records = validation_df[
        validation_df["record_status"] == "missing"
    ].copy()

    unsupported_unmatched = validation_df[
        validation_df["record_status"] == "unsupported_unmatched"
    ].copy()

    discrepant_records = validation_df[
        validation_df["record_status"] == "discrepant"
    ].copy()

    fully_correct_records = validation_df[
        validation_df["record_status"] == "fully_correct"
    ].copy()

    duplicate_extracted_records["record_status"] = "unsupported_duplicate"

    N_REF = int(len(reference_df))
    N_EXT = int(len(extracted_df))
    N_ALIGNED = int(aligned_mask.sum())
    N_MISSING = int(len(missing_records))
    N_UNSUPPORTED_UNMATCHED = int(len(unsupported_unmatched))
    N_DUPLICATE_EXTRAS = int(len(duplicate_extracted_records))
    N_UNSUPPORTED = N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS
    N_DISCREPANT = int(len(discrepant_records))
    N_CORRECT = int(len(fully_correct_records))

    completeness = N_ALIGNED / N_REF if N_REF else 0.0
    missing_rate = N_MISSING / N_REF if N_REF else 0.0
    precision = N_CORRECT / N_EXT if N_EXT else 0.0
    recall = N_CORRECT / N_REF if N_REF else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )
    unsupported_rate = N_UNSUPPORTED / N_EXT if N_EXT else 0.0
    discrepancy_rate = N_DISCREPANT / N_ALIGNED if N_ALIGNED else 0.0

    aligned_df = validation_df[aligned_mask].copy()

    field_accuracy_among_aligned = {
        "Concept Value": float(
            aligned_df["Concept Value_match"].mean()
        ) if N_ALIGNED else 0.0,

        "Publication Restricted": float(
            aligned_df[
                "Publication Restricted_match"
            ].mean()
        ) if N_ALIGNED else 0.0
    }

    correct_primary_field_instances = int(
        aligned_df[
            PRIMARY_MATCH_COLUMNS
        ].sum().sum()
    )

    evaluated_primary_field_instances = int(
        N_ALIGNED
        * len(PRIMARY_MATCH_COLUMNS)
    )

    field_accuracy = (
        correct_primary_field_instances
        / evaluated_primary_field_instances
        if evaluated_primary_field_instances
        else 0.0
    )

    summary = {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH_ID,
        "reference_records": N_REF,
        "extracted_records": N_EXT,
        "aligned_records": N_ALIGNED,
        "fully_correct_records": N_CORRECT,
        "discrepant_records": N_DISCREPANT,
        "field_accuracy":
            round(field_accuracy, 4),
        "missing_records": N_MISSING,
        "unsupported_extracted_records": N_UNSUPPORTED,
        "unsupported_unmatched_records": N_UNSUPPORTED_UNMATCHED,
        "unsupported_duplicate_records": N_DUPLICATE_EXTRAS,
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "completeness": round(completeness, 4),
        "missing_rate": round(missing_rate, 4),
        "record_precision_exact": round(precision, 4),
        "record_recall_exact": round(recall, 4),
        "record_f1_exact": round(f1, 4),
        "unsupported_rate": round(unsupported_rate, 4),
        "discrepancy_rate_among_aligned": round(discrepancy_rate, 4),
        "field_accuracy_among_aligned": {
            k: round(v, 4)
            for k, v in field_accuracy_among_aligned.items()
        },
        "schema_validity": schema_validity,
        "content_evaluable": True,
        "schema_diagnostics": schema_diagnostics,
        "matching_key_fields": MATCHING_FIELDS,
        "comparison_rules": {
            "identity_alignment": (
                "Normalised Section + Concept Name; value fields excluded"
            ),
            "primary_correctness": (
                "Exact source-string preservation for all requested fields"
            ),
            "concept_value_normalised_comparison": (
                "Diagnostic only"
            ),
            "concept_value_html_content_comparison": (
                "Diagnostic only"
            )
        },
        "normalisation_note": (
            "Normalisation is applied only to matching/diagnostic copies. "
            "The preserved Branch A response is not modified."
        ),
        "input_provenance": input_provenance
    }

    print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 13. Validation integrity checks and exports
# ------------------------------------------------------------

if content_evaluable:
    assert N_ALIGNED + N_MISSING == N_REF
    assert N_ALIGNED + N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS == N_EXT
    assert N_CORRECT + N_DISCREPANT == N_ALIGNED

    validation_df.to_csv(
        OUTPUT_DIR / "D4_branch_A_validation_detailed.csv",
        index=False
    )
    missing_records.to_csv(
        OUTPUT_DIR / "D4_branch_A_missing_records.csv",
        index=False
    )
    unsupported_unmatched.to_csv(
        OUTPUT_DIR / "D4_branch_A_unsupported_unmatched_records.csv",
        index=False
    )
    duplicate_extracted_records.to_csv(
        OUTPUT_DIR / "D4_branch_A_unsupported_duplicate_records.csv",
        index=False
    )
    discrepant_records.to_csv(
        OUTPUT_DIR / "D4_branch_A_discrepant_records.csv",
        index=False
    )
    fully_correct_records.to_csv(
        OUTPUT_DIR / "D4_branch_A_fully_correct_records.csv",
        index=False
    )

    field_summary_rows = []

    for field in PRIMARY_CORRECTNESS_FIELDS:
        match_column = f"{field}_match"

        correct_aligned = int(
            aligned_df[match_column].sum()
        )

        incorrect_aligned = int(
            N_ALIGNED - correct_aligned
        )

        field_summary_rows.append({
            "field": field,
            "aligned_records_evaluated":
                N_ALIGNED,
            "correct_values_among_aligned":
                correct_aligned,
            "incorrect_values_among_aligned":
                incorrect_aligned,
            "accuracy_among_aligned": (
                round(
                    correct_aligned / N_ALIGNED,
                    4
                )
                if N_ALIGNED
                else 0.0
            )
        })

    field_summary = pd.DataFrame(
        field_summary_rows
    )

    field_summary.to_csv(
        OUTPUT_DIR
        / "D4_branch_A_field_error_summary.csv",
        index=False
    )

    html_diagnostics = pd.DataFrame([{
        "aligned_records": N_ALIGNED,
        "concept_value_exact_matches": int(
            aligned_df["Concept Value_match"].sum()
        ),
        "concept_value_normalised_matches": int(
            aligned_df["Concept Value_normalised_match"].sum()
        ),
        "concept_value_content_matches": int(
            aligned_df["Concept Value_content_match"].sum()
        )
    }])

    html_diagnostics.to_csv(
        OUTPUT_DIR / "D4_branch_A_html_comparison_diagnostics.csv",
        index=False
    )

    with open(
        OUTPUT_DIR / "D4_branch_A_validation_summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("Validation integrity checks passed.")
    print("Content-validation artefacts saved.")
else:
    print(
        "Structural-failure artefacts were already saved. "
        "No content-level files were fabricated."
    )


In [ ]:
# ------------------------------------------------------------
# 14. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        files.download(output_file)